<a href="https://colab.research.google.com/github/rlp-jym/datatalksclub-zoomcamp-stock-markets-analytics-cohort-2026/blob/main/Module_1_Homework_(2026_cohort).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Setup

In [1]:
import requests, yfinance as yf, pandas as pd, duckdb
from bs4 import BeautifulSoup
from datetime import date

# Question 1: Which year had the highest number of additions (starting from 2020)?

In [2]:
url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x664) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.text, 'html.parser')
tables = soup.find_all('table')
sp500List = pd.read_html(str(tables[0]))[0]
sp500List = sp500List[['Symbol', 'Security', 'Date added']]

sp500List.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 503 entries, 0 to 502
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Symbol      503 non-null    object
 1   Security    503 non-null    object
 2   Date added  503 non-null    object
dtypes: object(3)
memory usage: 11.9+ KB


/tmp/ipykernel_42897/39050421.py:6: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  sp500List = pd.read_html(str(tables[0]))[0]


In [3]:
duckdb.sql("""
  select
    year("Date added"::date) as yearAdded
    , count() as count
  from sp500List
  group by yearAdded
  having yearAdded >= 2020
  order by count desc
""")

┌───────────┬───────┐
│ yearAdded │ count │
│   int64   │ int64 │
├───────────┼───────┤
│      2025 │    18 │
│      2024 │    16 │
│      2023 │    15 │
│      2022 │    15 │
│      2026 │    13 │
│      2021 │    10 │
│      2020 │    10 │
└───────────┴───────┘

Answer: 2025

# Question 2: How many indexes (out of 10) have better year-to-date returns than the US (S&P 500) as of August 21, 2026?



In [4]:
tickers = ['^GSPC', '000001.SS', '^HSI', '^AXJO', '^NSEI', '^GSPTSE', '^GDAXI', '^FTSE', '^N225', '^MXX', '^BVSP']
data = yf.download(tickers, start="2026-01-01", end="2026-08-22", progress=False, auto_adjust=True)
indixes = data['Close'].stack().reset_index()
indixes.columns = ['date', 'index', 'close']

print(indixes.info())
print("=" * 50)
print(indixes.date.min())
print(indixes.date.max())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1751 entries, 0 to 1750
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   date    1751 non-null   datetime64[ns]
 1   index   1751 non-null   object        
 2   close   1751 non-null   float64       
dtypes: datetime64[ns](1), float64(1), object(1)
memory usage: 41.2+ KB
None
2026-01-01 00:00:00
2026-08-21 00:00:00


In [5]:
duckdb.sql("""
  with
  prep as (
    select
      index
      , round(argmin(close, date), 2) as yearOpen
      , round(argmax(close, date), 2) as yearClose
    from indixes
    group by index
  ),
  chg as (
    select *
      , round((yearClose / yearOpen - 1) * 100, 2) as ytdChg
    from prep
    order by ytdChg desc
  )
  select *
  from chg
  where ytdChg >= (
    select ytdChg
    from chg
    where index = '^GSPC'
    )
""")

┌─────────┬──────────┬───────────┬────────┐
│  index  │ yearOpen │ yearClose │ ytdChg │
│ varchar │  double  │  double   │ double │
├─────────┼──────────┼───────────┼────────┤
│ ^N225   │  51832.8 │  66016.36 │  27.36 │
│ ^GSPTSE │  31883.4 │   36620.2 │  14.86 │
│ ^GSPC   │  6858.47 │   7674.37 │   11.9 │
└─────────┴──────────┴───────────┴────────┘

Answer: 2

# Question 3: Median drawdown (in %) of significant market corrections in the S&P 500 index



In [6]:
tickers = ['^GSPC']
data = yf.download(tickers, period='max', progress=False, auto_adjust=True)
sp500 = data['Close'].stack().reset_index()
sp500.columns = ['date', 'index', 'close']

sp500.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24780 entries, 0 to 24779
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   date    24780 non-null  datetime64[ns]
 1   index   24780 non-null  object        
 2   close   24780 non-null  float64       
dtypes: datetime64[ns](1), float64(1), object(1)
memory usage: 580.9+ KB


In [7]:
grouped = duckdb.sql("""
  with
  rolling_ath as (
    select
      date::date as date
      , close
      , max(close) over (
          order by date asc
          rows between unbounded preceding and current row)
          as rollingATH
    from sp500
  ),
  with_correction as (
    select *
      , (close / rollingATH - 1) * 100 as correction
      , case
          when close == rollingATH then 1
          else 0 end as is_ath
    from rolling_ath
  ),
  ath_grouped as (
    select *
      , sum(is_ath) over (order by date asc) as athGroup
    from with_correction
  )

  select
    athGroup
    , min(date) as athDateStart
    , count(*) as daysSinceATH
    , min(correction) as correction
  from ath_grouped
  group by athGroup
  order by athGroup asc
""").df()

print(grouped.head())
print("=" * 57)
print(f"Median: {grouped[grouped.correction < -5].correction.median():.0f}%")

   athGroup athDateStart  daysSinceATH  correction
0       1.0   1927-12-30             1    0.000000
1       2.0   1928-01-03            46   -4.560808
2       3.0   1928-03-09             1    0.000000
3       4.0   1928-03-12             3   -0.444444
4       5.0   1928-03-15             1    0.000000
Median: -8%


Answer: -8%

# Question 4: Calculate the median 2-day percentage change in stock prices following positive earnings surprise days.

In [8]:
amzn = 'AMZN'
ticker_obj = yf.Ticker(amzn)
amznEPS = ticker_obj.get_earnings_dates()
amznEPS = amznEPS.reset_index()
print(amznEPS.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25 entries, 0 to 24
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype                           
---  ------         --------------  -----                           
 0   Earnings Date  25 non-null     datetime64[ns, America/New_York]
 1   EPS Estimate   25 non-null     float64                         
 2   Reported EPS   24 non-null     float64                         
 3   Surprise(%)    24 non-null     float64                         
dtypes: datetime64[ns, America/New_York](1), float64(3)
memory usage: 932.0 bytes
None


In [9]:
data = yf.download(amzn, period='max', progress=False, auto_adjust=True)
amznClose = data['Close'].stack().reset_index()
amznClose.columns = ['date', 'index', 'close']
print(amznClose.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7366 entries, 0 to 7365
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   date    7366 non-null   datetime64[ns]
 1   index   7366 non-null   object        
 2   close   7366 non-null   float64       
dtypes: datetime64[ns](1), float64(1), object(1)
memory usage: 172.8+ KB
None


In [10]:
amznEPSClean = duckdb.sql("""
select
  "Earnings Date"::date as date
  , "Surprise(%)" as surprise
from amznEPS
""")

amznCloseClean = duckdb.sql("""
select
  date::date as date
  , lag(close, 1) over (order by date asc) as closePre
  , lead(close, 2) over (order by date asc) as closePost
from amznClose
""")

In [11]:
amznJoined = duckdb.sql("""
select
  a.date, b.surprise, a.closePre, a.closePost
  , (a.closePost / a.closePre - 1) * 100 as pctChange
from amznCloseClean a
left join amznEPSClean b
  on a.date = b.date
where b.surprise is not null
  and b.surprise > 0
""").df()

print(f"Median: {amznJoined.pctChange.median():.02f}%")

Median: 0.35%


Answer: 0.35%